# Visualização exploratória — renda por setor censitário

Este notebook serve para **pré-visualizar** a distribuição espacial da renda no nível dos setores censitários antes de definir o estilo final das figuras do piloto.

O gradiente representa `RENDA_MED_SETOR`. Os limites e nomes dos bairros são usados apenas como referência territorial. A visualização **não é salva automaticamente** em PNG.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualization import preview_sector_income_with_neighborhood_labels

DATA_DIR = PROJECT_ROOT / "data" / "censo_2022"
GPKG_PATH = DATA_DIR / "bairros_setores_renda.gpkg"
BOUNDARY_PATH = DATA_DIR / "limite_poa.gpkg"

print("Raiz do projeto:", PROJECT_ROOT)
print("GeoPackage:", GPKG_PATH)

In [ ]:
if not GPKG_PATH.exists():
    raise FileNotFoundError(
        "O GeoPackage ainda não existe. Execute primeiro: "
        "python scripts/downloads.py"
    )

setores = gpd.read_file(
    GPKG_PATH,
    layer="setores_renda",
)

bairros = gpd.read_file(
    GPKG_PATH,
    layer="bairros",
)

limite = None
if BOUNDARY_PATH.exists():
    limite = gpd.read_file(
        BOUNDARY_PATH,
        layer="limite_poa",
    )

print(f"Setores carregados: {len(setores)}")
print(f"Bairros carregados: {len(bairros)}")
print("CRS dos setores:", setores.crs)
print("Setores sem renda:", setores["RENDA_MED_SETOR"].isna().sum())

In [ ]:
# Inspeção da distribuição da renda setorial
setores[["CD_SETOR", "RENDA_MED_SETOR"]].sort_values(
    "RENDA_MED_SETOR"
).reset_index(drop=True)

In [ ]:
# Parâmetros exploratórios: altere livremente para comparar alternativas.
CMAP = "YlOrRd"
FIGSIZE = (16, 16)
LABEL_FONTSIZE = 5.5

fig, ax = preview_sector_income_with_neighborhood_labels(
    sectors=setores,
    neighborhoods=bairros,
    value_column="RENDA_MED_SETOR",
    name_column="NM_BAIRRO",
    title="Distribuição da renda média por setor censitário — Porto Alegre",
    cmap=CMAP,
    boundary=limite,
    figsize=FIGSIZE,
    label_fontsize=LABEL_FONTSIZE,
)

# A última expressão exibe a figura no notebook.
# Nenhum arquivo é salvo por esta célula.
fig

## O que observar antes de definir o mapa final

- heterogeneidade de renda **dentro de um mesmo bairro**;
- setores sem valor de renda;
- efeito de valores extremos sobre o gradiente;
- legibilidade dos nomes dos bairros sobre uma malha mais detalhada;
- necessidade de usar escala contínua, quantis ou outra classificação na versão final.

Os bairros são apenas referência visual; a unidade socioeconômica representada é o **setor censitário**.